# NephroAI: Chronic Kidney Disease Prediction System
### Standardized Clinical Machine Learning System & Pathology Diagnosis

This notebook documents the standardized, publication-grade development of **NephroAI**, a machine learning prediction system for Chronic Kidney Disease (CKD) using an optimized clinical K-Nearest Neighbors (KNN) pipeline.

### 🔬 The Core Pathology Challenges Addressed:
1. **Clinical Dataset Integration:** Cleaned and combined two heterogeneous clinical datasets (UCI CKD with 400 samples/25 features, and Risk Factor CKD with 200 samples/29 features) containing non-standard feature schemas and ranges.
2. **Elimination of Data Leakage (The Cardinal Sin of ML):** Corrected a severe bug where imputation, scaling, and feature selection were computed on the full dataset before train/test splitting. Preprocessing and feature selection are now strictly fit *only* on the training split, ensuring mathematically sound validation.
3. **Production-Grade Pipeline Serialization:** Built a unified Scikit-Learn `Pipeline` object that serializes imputation, scaling, feature selection, and classification into a single `.joblib` deployment artifact.
4. **Interactive Dashboard:** Deployed a beautiful Glassmorphism local web interface in Flask for real-time pathology screening.

## Phase 1: Imports & Configurations

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
from joblib import dump, load

# Set plot styles
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Paths & Parameters
PROCESSED_DATA_PATH = "../data/processed/ckd_merged_corrected.csv"
RANDOM_STATE = 42
TEST_SIZE = 0.20

## Phase 2: Exploratory Data Analysis & Null Visualizations
Let's load the preprocessed merged dataset and inspect the missing values.

In [ ]:
if not os.path.exists(PROCESSED_DATA_PATH):
    # Fallback to local execution if running from root
    PROCESSED_DATA_PATH = "data/processed/ckd_merged_corrected.csv"

df = pd.read_csv(PROCESSED_DATA_PATH)
print(f"Loaded merged processed dataset. Shape: {df.shape}")
df.head()

In [ ]:
# Visualize missing value densities per feature
null_counts = df.isnull().sum()
null_percentages = (null_counts / len(df)) * 100
null_df = pd.DataFrame({"Null Count": null_counts, "Percentage": null_percentages}).sort_values(by="Null Count", ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x=null_df.index[:15], y=null_df["Percentage"][:15], palette="viridis")
plt.title("Top 15 Features with Missing Values (% Density)", fontsize=14, fontweight="bold")
plt.ylabel("Missing Percentage (%)", fontsize=12)
plt.xlabel("Clinical Attributes", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Phase 3: Zero Leakage Training Split

We now split the target variable `affected` and features `X`. Crucially, to prevent **data leakage**, we perform the train/test split **before** fitting the imputer, standardizer, and feature selection module.

In [ ]:
target_col = "affected"
X = df.drop(columns=[target_col]).copy()
y = df[target_col].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)
print(f"Train set shape: {X_train.shape}")
print(f"Test set shape:  {X_test.shape}")

## Phase 4: Leakage-Free Preprocessing & Feature Selection

We learn the optimal parameters from `X_train` only and apply them to `X_test` to verify true generalization metrics.

In [ ]:
# 1. Imputation
imputer = IterativeImputer(max_iter=20, random_state=RANDOM_STATE)
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

# 2. Scaling (Consolidated to a single mathematical StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled = scaler.transform(X_test_imp)

# 3. Feature Selection using RandomForest
rf = RandomForestClassifier(
    n_estimators=500,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight="balanced_subsample"
)
rf.fit(X_train_scaled, y_train)

selector = SelectFromModel(rf, threshold="median", prefit=True)
support_mask = selector.get_support()
selected_features = np.array(X.columns)[support_mask].tolist()

X_train_sel = X_train_scaled[:, support_mask]
X_test_sel = X_test_scaled[:, support_mask]

print(f"Selected {len(selected_features)} features during training:")
print(selected_features)

## Phase 5: Fast Grid Search & Evaluation
We run a grid search only on the KNN parameters using the preprocessed train split, ensuring fast and robust cross-validation.

In [ ]:
param_grid = {
    "n_neighbors": [3, 5, 7, 9, 11],
    "metric": ["euclidean", "manhattan"],
    "weights": ["uniform", "distance"],
    "leaf_size": [20, 30],
    "algorithm": ["auto", "kd_tree"]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train_sel, y_train)
print("Best parameters found:", grid.best_params_)
print(f"Best CV accuracy: {grid.best_score_*100:.2f}%")

In [ ]:
# Evaluate on Independent Test Set (Zero Leakage!)
best_knn = grid.best_estimator_
y_pred = best_knn.predict(X_test_sel)

print(f"Leakage-Free Test Set Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred, digits=4))

# Display confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Healthy", "CKD"])
disp.plot(cmap="Blues")
plt.title("Independent Test Confusion Matrix", fontsize=13, fontweight="bold")
plt.show()

## Phase 6: Unified Pipeline Refit & Serialization

For production deployment, we refit the entire unified pipeline configuration on the **full dataset** and serialize it directly.

In [ ]:
production_pipeline = Pipeline([
    ("imputer", IterativeImputer(max_iter=20, random_state=RANDOM_STATE)),
    ("scaler", StandardScaler()),
    ("feature_selector", SelectFromModel(
        RandomForestClassifier(
            n_estimators=500,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced_subsample"
        ),
        threshold="median"
    )),
    ("model", KNeighborsClassifier(**grid.best_params_))
])

production_pipeline.fit(X, y)

# Save production pipeline
MODEL_PATH = "../models/ckd_knn_pipeline.joblib"
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
dump(production_pipeline, MODEL_PATH)
print(f"Successfully saved the production pipeline to: {MODEL_PATH}")

---